In [1]:
import os
import pandas as pd
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')
import numpy as np

## Baseline Creation

In [ ]:
BaselineFolder=r" ACES"

In [ ]:
Baselinefile_list = []

for root, dirs, files in os.walk(BaselineFolder):
    for f in files:
        if (".xlsx") in f and ("CARDONE") in f:
            Baselinefile_list.append(os.path.join(root, f))

Baselinefile_list

In [ ]:
# Update the BaseKeyColumns and BaseValidColumns as needed
BaseKeyColumns=['Make','Model','Year','Position']
BaseValidColumns=BaseKeyColumns+['Product','PartNumber']
BaseValidColumns

In [ ]:
df_baseline = pd.DataFrame(columns=["Sl.No"])
df_baseline

for i in Baselinefile_list:
    df = pd.read_excel(i)
    df_baseline=pd.concat([df_baseline,df],ignore_index=True)

df_baseline=df_baseline[BaseValidColumns].drop_duplicates()
df_baseline['Source']="CARDONE"
df_baseline['Year']=df_baseline['Year'].astype(str).str[:4]
df_baseline['Model']=df_baseline['Model'].astype(str)
df_baseline

## Target Creation

In [ ]:
TargetFolder=r"Target ACES"

In [ ]:
Targetfile_list = []

for root, dirs, files in os.walk(TargetFolder):
    for f in files:
        if (".xlsx") in f and ("linkages") in f:
            Targetfile_list.append(os.path.join(root, f))

Targetfile_list

In [ ]:
# Update the TargetKeyColumns and TargetValidColumns as needed
TargetKeyColumns=['Make','Model','Year','Position']
TargetValidColumns=TargetKeyColumns+['ProductGroup','ArticleNumber']
TargetValidColumns

['Make', 'Model', 'Year', 'Position', 'ProductGroup', 'ArticleNumber']

In [ ]:
df_target = pd.DataFrame(columns=["Sl.No"])
df_target
for i in Targetfile_list:
    df = pd.read_excel(i)
    df_target=pd.concat([df_target,df],ignore_index=True)

df_target=df_target[TargetValidColumns].drop_duplicates()
df_target['Source']="JNP"
df_target['Year']=df_target['Year'].astype(str).str[:4]
df_target['Model']=df_target['Model'].astype(str)
df_target

In [ ]:
df_baseline['Available_in_JNP']=(df_target.groupby(TargetKeyColumns)['ArticleNumber']
                                 .apply(list)
                                 .reindex(df_baseline.set_index(BaseKeyColumns).index)
                                 .apply(lambda x: x if isinstance(x, list) else "Not Available in JNP")
                                 .values
                                 )
df_baseline

In [ ]:
df_target['Available_in_CARDONE']=(df_baseline.groupby(BaseKeyColumns)['PartNumber']
                                 .apply(list)
                                 .reindex(df_target.set_index(TargetValidColumns).index)
                                 .apply(lambda x: x if isinstance(x, list) else "Not Available in CARDONE")
                                 .values
                                 )
df_target

In [13]:
OFolder=r"C:\Users\vikram.vadhirajan\OneDrive - Trico\FBG\06_SPD_Tasks\Catalog_Team_Requests\Comparison"

In [14]:
with pd.ExcelWriter(OFolder+'\\'+f'CALIPER_CARDONE_JNP_Comparison_1.xlsx') as writer:  # doctest: +SKIP
    df_target.to_excel(writer,index=False, sheet_name='JNP_CALIPER_List')
    df_baseline.to_excel(writer,index=False, sheet_name='CARDONE_CALIPER_List')